# Sequence Preprocessing

In [238]:
#Libraries needed: 

# Biopython: parse, manipulate, and analyze biological data
from Bio import SeqIO,AlignIO
from Bio.Align import MultipleSeqAlignment
from Bio.PDB import PDBParser

# Python libraries:
import random as rd
import math
from collections import Counter

#sequence alignment
import subprocess

#Data Manipulation:
import pandas as pd
import numpy as np

### Read in data and filter

In [239]:
#read the fasta file

def open_file(file,format):
    """The function parses downloaded reference sequences"""

    record = list(SeqIO.parse(file, format))
    return record

var_protein = open_file("virus_protein.fasta", format = "fasta") #2000 variant sequences
init_protein = open_file("wuhan_spike_YP_009724390.1.fasta", format = "fasta") #reference sequence

#var_protein
#init_protein

for rec in var_protein[ :3]: #just checking it printed the sequences
    print(rec.id)
    print(str(rec.seq[ :10]))
    print("-" * 60)

#MFVFLVLLPL
#MFVFLVLLPL
#MFVFLVLLPL


for rec in init_protein: #just checking it printed the sequences
    print(rec.id)
    print(str(rec.seq[ :10]))
    print("-" * 60)

#MFVFLVLLPL

WKQ99033.1
MFVFLVLLPL
------------------------------------------------------------
UEZ51942.1
MFVFLVLLPL
------------------------------------------------------------
XRW97038.1
MFVFLVLLPL
------------------------------------------------------------
YP_009724390.1
MFVFLVLLPL
------------------------------------------------------------


In [240]:
#sanity checks
#print(len(var_protein),len(init_protein))     # should be ~2000 Sars-COV-2 protein sequences; should be 1: Wuhan Reference Sequence
#print(var_protein[0].id,len(var_protein[0].seq),init_protein[0].id,len(init_protein[0].seq)) # WKQ99033.1 1270 YP_009724390.1 1273

lengths = [len(rec.seq) for rec in var_protein]
min_len = min(lengths)
max_len = max(lengths)
#print(min_len,max_len) # 1251 1275

In [241]:
#We need to filter out the sequences with large chunks of X's because those are unknown amino acids, 
#cannot predict mutations with large proportions of unknown amino acids

def normalize(seq):
    norm_seq = seq.upper().replace("*", "").replace("-", "").strip() #accounts for unnecessary characters like stop symbols
    return norm_seq
    
valid_aa = {
    "A","R","N","D","C","Q","E","G","H","I",
    "L","K","M","F","P","S","T","W","Y","V"
}

def valid_aa_seq(seq,max_x = 0.20,min_std_aa = 0.80):
    
    seq = normalize(seq)
    length = len(seq)

    if length == 0:
        return False

    x_frac = seq.count("X")/length

    aa_count = sum(aa in valid_aa for aa in seq)
    aa_frac = aa_count/length

    if x_frac > max_x: #checks if counts of "X's" are larger than threshold 0.02
        return False

    if aa_frac < min_std_aa:
        return False

    return True
  
#valid_aa_seq("XCDXFGHIKX")

#apply filtering function to 2000 sequences
max_x = 0.2
min_std_aa = 0.8
keep_seq = []
out_seq = []

for rec in var_protein:
    if valid_aa_seq(rec.seq,max_x,min_std_aa):
        keep_seq.append(rec)
        
    else:
         out_seq.append(rec)

print("Valid sequences:", len(keep_seq))
print("Invalid sequences:", len(out_seq))



Valid sequences: 1938
Invalid sequences: 62


In [242]:
#examine which sequences are kept
def seqrecords_to_df(records):
    """
    Convert a list of Bio.SeqRecord objects into a pandas DataFrame.
    """
    rows = []

    for rec in records:
        seq = normalize(str(rec.seq))
        rows.append({
            "id": rec.id,
            "description": rec.description,
            "sequence": seq,
            "length": len(seq),
            "X_count": seq.count("X"),
            "X_fraction": seq.count("X") / len(seq) if len(seq) > 0 else 0
        })

    return pd.DataFrame(rows)

keep_seq_df = seqrecords_to_df(keep_seq)
keep_seq_df.to_csv("keep_sequences.csv", index=False)

In [243]:
#Remove Duplicate Sequences under different Ids
unique_set = {}

for rec in keep_seq:
    norm_seq = normalize(rec.seq) 
    if norm_seq not in unique_set: 
        unique_set[norm_seq] = rec

unique_rec = list(unique_set.values())

seqs = [normalize(rec.seq) for rec in unique_rec]
# print(len(seqs), len(set(seqs))) #set says give me unique sequence strings only, numbers should match if duplicates removed

valid_df = seqrecords_to_df(unique_rec)
valid_df.to_csv("valid_seq.csv", index=False)
print("Valid sequences:", len(valid_df))

Valid sequences: 1058


In [244]:
#unique_set of sequences we need 50
rd.seed(42)

sample_protein = rd.sample(unique_rec,50)
print(len(sample_protein))

seq_strings = [normalize(rec.seq) for rec in sample_protein]

# print("Total sequences:", len(seq_strings))
# print("Unique sequences:", len(set(seq_strings))) #some identical sequences

counts = Counter(seq_strings)
duplicates = {seq: c for seq, c in counts.items() if c > 1}

len(duplicates)

protein_df = seqrecords_to_df(sample_protein)
protein_df.to_csv("protein_df.csv", index=False)
print("Subsetted Protein Sequences:" ,len(protein_df))

50
Subsetted Protein Sequences: 50


In [245]:
#save as fasta file so its reproducible 
def sample_spikes(sample,file_name,format):
    """This function saves the sample set of proteins to a fasta file"""
    
    SeqIO.write(sample,file_name,format)
    return file_name
    

sample_spikes(sample_protein,"sample_protein50.fasta",format = "fasta")

#sanity checks 
protein_50 = open_file("sample_protein50.fasta", format = "fasta")
# print(len(protein_50)) #~50 random sequences

lengths = [len(rec.seq) for rec in protein_50]
min_len = min(lengths)
max_len = max(lengths)

# print(min_len)
# print(max_len)

In [246]:
#build data frame of 50 random sequences and reference wuhan strain 

#1. combine reference Wuhan strain [0] with the 50 sample sequences
ref_record = init_protein[0]  # wuhan spike record
ref_record.id = "WUHAN_REF"
combined = [ref_record] + sample_protein 

#2. save in fasta file
combined_50ref = sample_spikes(combined,"combined_50ref.fasta",format = "fasta") 

In [247]:
input_fasta = "combined_50ref.fasta"
output_fasta = "combined_50ref_aligned.fasta"

with open(output_fasta, "w") as outfile:

    subprocess.run(
        [
            r"C:\Users\bphul\Downloads\mafft-7.526-win64-signed\mafft-win\mafft.bat",
            "--auto",
            input_fasta
        ],
        stdout=outfile,
        text=True
    )

print("Alignment complete")

Alignment complete


In [248]:
#Check alignment
combined_50ref_aligned = open_file("combined_50ref_aligned.fasta",format = "fasta")

#sanity check
print(len(combined_50ref_aligned))            # should be 51
print(combined_50ref_aligned[0].id)           # Wuhan
print(len(combined_50ref_aligned[0].seq))     # alignment length wuhan
print(len(combined_50ref_aligned[1].seq))     # alignment length random sequence

51
WUHAN_REF
1280
1280


### Create Dataframe

In [249]:
data = {}
for rec in combined_50ref_aligned:
    data[rec.id] = list(str(rec.seq))

df_wide = pd.DataFrame.from_dict(data, orient="index")

In [250]:
#wide format dataframe with sequences stacked 
df = pd.DataFrame({
    "id": [rec.id for rec in combined_50ref_aligned],
    "seq": [list(str(rec.seq)) for rec in combined_50ref_aligned]
})
#df

#wide format that we want
seq_df = pd.DataFrame(df["seq"].tolist())

#put it together
seq_df.index = df["id"]
seq_df.index.name = "seq_id"

In [251]:
#change to long format 
wide = seq_df.copy()
#wide.head()

wide = wide.reset_index().rename(columns={"index":"seq_id"})
wide.head()

#melt: take each column and pour it down into rows
long = wide.melt(
    id_vars = "seq_id", #keep seq_id as is
    var_name = "aln_index", #take col names 0-1280 and put them in a new col called aln_index
    value_name = "sample_residue" #take the residue values and put them in a new col called sample_residue
)

In [252]:
#identify wuhan id and store it
wuhan_id = "WUHAN_REF"

#select the reference id
wuhan_row = seq_df.loc[wuhan_id] #give me row of reference sequence, we know sequence name is safer to use

#make a cheat sheet for reference
wuhan_map = wuhan_row.to_dict() #can identify what the wuhan residue at pos i 

# map for every seq look at the pos i and check the cheat sheet and write the wuhan letter in "wuhan_residue"
long["wuhan_residue"] = long["aln_index"].map(wuhan_map)
long = long[long["seq_id"] != "WUHAN_REF"].copy() #remove redundant wuhan vs wuhan rows


#how it should look 
#seq_id	aln_index	sample_residue	wuhan_residue
#S1      	0	          M    	        M
#S1	        1	          F         	F
#S1	        2	          I         	V

print(len(long))
#long.head(10)

64000


In [350]:
#make a copy of analysis dataset
seq1_df = long.copy()

### Add Features to Data Frame

In [351]:
#define classes for mutations

valid_aa = set("ARNDCQEGHILKMFPSTWYV") #valid amino acids counted for mutations

def mutations(ref,seq):

    ref = str(ref).upper()
    seq = str(seq).upper()
    
    # if unknown AA
    if ref == "X" or seq == "X":
        return None
    # if both gaps -> nonexistent structure
    if ref == "-" and seq == "-":
        return None

    # if aa not in valid set
    if (ref not in valid_aa and ref != "-") or (seq not in valid_aa and seq != "-"):
        return None 

# mutation identification: 
    
    # stable (same amino acid, both real) 
    if ref == seq and ref != "-":
        return 0
    
    # substitution (both real, different)
    if ref != "-" and seq != "-" and ref != seq:
        return 1
    
    # deletion (reference has AA, sample has gap)
    if ref != "-" and seq == "-":
        return 2
    
    # insertion (reference has gap, sample has AA)
    if ref == "-" and seq != "-":
        return 3

# print("mutation:",mutations("A","A"))  # get mutation = 0 
# print("mutation:" ,mutations("A","G")) # get mutation = 1
# print("mutation:",mutations("A","-"))  # get mutation = 2
# print("mutation:",mutations("-","A"))  # get mutation = 3 
# print("mutation:",mutations("-","-"))  # get mutation = None since both sequence nucleotides are missing

In [352]:
# use mutation function to identify mutations from dataframe long
seq1_df["mutation"] = seq1_df.apply(
    lambda r: mutations(r["wuhan_residue"],r["sample_residue"]),
    axis = 1 #axis = 1 compare within same row; axis = 0 summarize/compare in same column so it wouldnt work in this case and provide wrong mutations
)

mutation_seq1_df = seq1_df.copy()


missing = mutation_seq1_df["mutation"].isna().sum()
num_unknowns = (mutation_seq1_df["sample_residue"] == "X").sum()
tot_prop_missunk = mutation_seq1_df["mutation"].isna().sum() + (mutation_seq1_df["mutation"] == "Unknown").sum()
print(len(mutation_seq1_df),
      "missing:", missing," ",
      "Total unknown residues (X):", num_unknowns," "
      "Total prop of unknown and missing residues:", tot_prop_missunk/len(mutation_seq1_df)
     )

64000 missing: 2396   Total unknown residues (X): 2078  Total prop of unknown and missing residues: 0.0374375


In [353]:
#valid amino acids proportion per position
total_seqs = seq1_df["seq_id"].nunique()

pos_rates = (
    seq1_df
    .groupby("aln_index",as_index = False) #.groupby -> %>% group_by in R
    .agg(  # %>% summarize() in R
        valid_n = ("mutation",lambda x: x.notna().sum())
        
    )
)

pos_rates["valid_frac"] = pos_rates["valid_n"]/total_seqs

good_pos = pos_rates.loc[
    pos_rates["valid_frac"] >= 0.70,"aln_index"]

seq1_df = seq1_df[seq1_df["aln_index"].isin(good_pos)].copy()
seq1_df = seq1_df.merge(pos_rates, on = "aln_index",how = "left")

In [354]:
print(pos_rates[pos_rates["valid_frac"] < 0.7])

     aln_index  valid_n  valid_frac
16          16        3        0.06
17          17        3        0.06
18          18        3        0.06
19          19        3        0.06
218        218        7        0.14
219        219        7        0.14
220        220        7        0.14


In [355]:
len(seq1_df)

63650

In [356]:
#Target feature: Mutation Hotspots
seq1_df = seq1_df[seq1_df["wuhan_residue"].isin(valid_aa)].copy()

def hotspot_label(df):

    pos = (
        df.groupby("aln_index",as_index = False)
        .agg(
            mut_count = ("mutation",lambda s: s.isin([1,2,3]).sum()),
            valid_count = ("seq_id","nunique")
            
        )
    )

    pos["mutation_rate"] = pos["mut_count"]/pos["valid_count"]
    pos["hotspot"] = (pos["mut_count"] > 0 ).astype(int)
    return pos

pos_df = hotspot_label(seq1_df)
seq1_df = seq1_df.merge(pos_df[["aln_index","hotspot","mutation_rate"]], on = "aln_index", how = "left")


seq1_df.head(50)

,seq_id,aln_index,sample_residue,wuhan_residue,mutation,valid_n,valid_frac,hotspot,mutation_rate
0,UYE19200.1,0,M,M,0.0,50,1.0,1,0.04
1,UHX08132.1,0,M,M,0.0,50,1.0,1,0.04
2,UHS39707.1,0,M,M,0.0,50,1.0,1,0.04
3,XPK52150.1,0,M,M,0.0,50,1.0,1,0.04
4,UBO31061.1,0,M,M,0.0,50,1.0,1,0.04
5,WPM52290.1,0,-,M,2.0,50,1.0,1,0.04
6,UMY93497.1,0,M,M,0.0,50,1.0,1,0.04
7,UCO39187.1,0,M,M,0.0,50,1.0,1,0.04
8,UGO67261.1,0,M,M,0.0,50,1.0,1,0.04
9,UAV83603.1,0,M,M,0.0,50,1.0,1,0.04


In [367]:
print(seq1_df["hotspot"].value_counts()) # 6450/50 = 129 hotspots

hotspot
0    57200
1     6450
Name: count, dtype: int64


In [358]:
#create position of Wuhan ref sequence mapping
site_df = (
    seq1_df
    .groupby("aln_index", as_index=False)
    .agg(wuhan_residue=("wuhan_residue", "first"))
    .sort_values("aln_index")
)

#map onto site_df
site_df["is_real_residue"] = (
    site_df["wuhan_residue"].notna() &
    (site_df["wuhan_residue"] != "") &
    (site_df["wuhan_residue"] != "-")
)

site_df["wuhan_resnum"] = site_df["is_real_residue"].cumsum()
site_df.loc[~site_df["is_real_residue"], "wuhan_resnum"] = pd.NA

seq1_df = seq1_df.merge(site_df[["aln_index", "wuhan_resnum"]], on = "aln_index",how = "left")

In [359]:
seq1_df.head(10)

,seq_id,aln_index,sample_residue,wuhan_residue,mutation,valid_n,valid_frac,hotspot,mutation_rate,wuhan_resnum
0,UYE19200.1,0,M,M,0.0,50,1.0,1,0.04,1.0
1,UHX08132.1,0,M,M,0.0,50,1.0,1,0.04,1.0
2,UHS39707.1,0,M,M,0.0,50,1.0,1,0.04,1.0
3,XPK52150.1,0,M,M,0.0,50,1.0,1,0.04,1.0
4,UBO31061.1,0,M,M,0.0,50,1.0,1,0.04,1.0
5,WPM52290.1,0,-,M,2.0,50,1.0,1,0.04,1.0
6,UMY93497.1,0,M,M,0.0,50,1.0,1,0.04,1.0
7,UCO39187.1,0,M,M,0.0,50,1.0,1,0.04,1.0
8,UGO67261.1,0,M,M,0.0,50,1.0,1,0.04,1.0
9,UAV83603.1,0,M,M,0.0,50,1.0,1,0.04,1.0


### Add in Structural Features

In [360]:
#read in rsv.csv
rsa_df = pd.read_csv("RSA.csv")
print(rsa_df.columns)

#remove space before column name
rsa_df.columns = rsa_df.columns.str.strip() #<- strip removes whitespace before the col name
#print(rsa_df.columns)

#rename columns
rsa_df = rsa_df.rename(columns={
    "n": "wuhan_resnum",   
})
rsa_df["wuhan_resnum"] = rsa_df["wuhan_resnum"].astype(int)

# kept features that explain more of the structural features
rsa_df = rsa_df.drop(columns = ["id","seq","q3","p[q3_H]","p[q3_E]","p[q3_C]","phi","psi"])


# merge it back onto seq1_df
seq1_df = seq1_df.merge(
    rsa_df,
    on="wuhan_resnum",
    how="left"
)

Index(['id', ' seq', ' n', ' rsa', ' asa', ' q3', ' p[q3_H]', ' p[q3_E]',
       ' p[q3_C]', ' q8', ' p[q8_G]', ' p[q8_H]', ' p[q8_I]', ' p[q8_B]',
       ' p[q8_E]', ' p[q8_S]', ' p[q8_T]', ' p[q8_C]', ' phi', ' psi',
       ' disorder'],
      dtype='str')


In [339]:
seq1_df.head(10)

,seq_id,aln_index,sample_residue,wuhan_residue,mutation,valid_n,valid_frac,hotspot,mutation_rate,wuhan_resnum,...,q8,p[q8_G],p[q8_H],p[q8_I],p[q8_B],p[q8_E],p[q8_S],p[q8_T],p[q8_C],disorder
0,UYE19200.1,0,M,M,0.0,50,1.0,1,0.04,1.0,...,C,0.000175,0.000576,0.000018,0.000184,0.000401,0.00093,0.000894,0.996821,0.996505
1,UHX08132.1,0,M,M,0.0,50,1.0,1,0.04,1.0,...,C,0.000175,0.000576,0.000018,0.000184,0.000401,0.00093,0.000894,0.996821,0.996505
2,UHS39707.1,0,M,M,0.0,50,1.0,1,0.04,1.0,...,C,0.000175,0.000576,0.000018,0.000184,0.000401,0.00093,0.000894,0.996821,0.996505
3,XPK52150.1,0,M,M,0.0,50,1.0,1,0.04,1.0,...,C,0.000175,0.000576,0.000018,0.000184,0.000401,0.00093,0.000894,0.996821,0.996505
4,UBO31061.1,0,M,M,0.0,50,1.0,1,0.04,1.0,...,C,0.000175,0.000576,0.000018,0.000184,0.000401,0.00093,0.000894,0.996821,0.996505
5,WPM52290.1,0,-,M,2.0,50,1.0,1,0.04,1.0,...,C,0.000175,0.000576,0.000018,0.000184,0.000401,0.00093,0.000894,0.996821,0.996505
6,UMY93497.1,0,M,M,0.0,50,1.0,1,0.04,1.0,...,C,0.000175,0.000576,0.000018,0.000184,0.000401,0.00093,0.000894,0.996821,0.996505
7,UCO39187.1,0,M,M,0.0,50,1.0,1,0.04,1.0,...,C,0.000175,0.000576,0.000018,0.000184,0.000401,0.00093,0.000894,0.996821,0.996505
8,UGO67261.1,0,M,M,0.0,50,1.0,1,0.04,1.0,...,C,0.000175,0.000576,0.000018,0.000184,0.000401,0.00093,0.000894,0.996821,0.996505
9,UAV83603.1,0,M,M,0.0,50,1.0,1,0.04,1.0,...,C,0.000175,0.000576,0.000018,0.000184,0.000401,0.00093,0.000894,0.996821,0.996505


In [361]:
# Hydrophobicity

#create dictionary of KD-values for amino acids
kd = {
    'A': 1.8, 'R': -4.5, 'N': -3.5, 'D': -3.5,
    'C': 2.5, 'Q': -3.5, 'E': -3.5, 'G': -0.4,
    'H': -3.2, 'I': 4.5, 'L': 3.8, 'K': -3.9,
    'M': 1.9, 'F': 2.8, 'P': -1.6, 'S': -0.8,
    'T': -0.7, 'W': -0.9, 'Y': -1.3, 'V': 4.2
}

kd_df = (
    seq1_df.copy().sort_values("aln_index")
           .groupby("aln_index", as_index = False)
           .agg(wuhan_residue = ("wuhan_residue","first"),
                wuhan_resnum = ("wuhan_resnum","first")
               )
)

kd_df["kd_ref"] = kd_df["wuhan_residue"].map(kd) #convert wuhan residue into KD-score

#sliding windows need to go through real residues so I need exclude the gaps positions from wuhan sequence (ungapped)
hydro_df = kd_df[kd_df["wuhan_resnum"].notna()].copy() #keep valids
hydro_df["wuhan_resnum"] = hydro_df["wuhan_resnum"].astype(int)
hydro_df = hydro_df.sort_values("wuhan_resnum")

#shorter window (9) and longer windows (15,21)
for W in [9,15,21]: #look at W neighbors
    hydro_df[f"kd_win{W}"] = ( #<- called f-string -> kd_win9 for W = 9, creating columns
        hydro_df["kd_ref"]
        .rolling(window=W, center=True, min_periods=1) # center = center the window on the current residue, min_periods = near the edges, still compute average even if full window isn’t available
        .mean()
    )

seq1_df = seq1_df.merge(hydro_df[["wuhan_resnum","kd_ref","kd_win9","kd_win15", "kd_win21"]], 
                               on = "wuhan_resnum",
                               how = "left"
                              )

In [341]:
seq1_df.head(10)

,seq_id,aln_index,sample_residue,wuhan_residue,mutation,valid_n,valid_frac,hotspot,mutation_rate,wuhan_resnum,...,p[q8_B],p[q8_E],p[q8_S],p[q8_T],p[q8_C],disorder,kd_ref,kd_win9,kd_win15,kd_win21
0,UYE19200.1,0,M,M,0.0,50,1.0,1,0.04,1.0,...,0.000184,0.000401,0.00093,0.000894,0.996821,0.996505,1.9,3.1,3.4125,3.063636
1,UHX08132.1,0,M,M,0.0,50,1.0,1,0.04,1.0,...,0.000184,0.000401,0.00093,0.000894,0.996821,0.996505,1.9,3.1,3.4125,3.063636
2,UHS39707.1,0,M,M,0.0,50,1.0,1,0.04,1.0,...,0.000184,0.000401,0.00093,0.000894,0.996821,0.996505,1.9,3.1,3.4125,3.063636
3,XPK52150.1,0,M,M,0.0,50,1.0,1,0.04,1.0,...,0.000184,0.000401,0.00093,0.000894,0.996821,0.996505,1.9,3.1,3.4125,3.063636
4,UBO31061.1,0,M,M,0.0,50,1.0,1,0.04,1.0,...,0.000184,0.000401,0.00093,0.000894,0.996821,0.996505,1.9,3.1,3.4125,3.063636
5,WPM52290.1,0,-,M,2.0,50,1.0,1,0.04,1.0,...,0.000184,0.000401,0.00093,0.000894,0.996821,0.996505,1.9,3.1,3.4125,3.063636
6,UMY93497.1,0,M,M,0.0,50,1.0,1,0.04,1.0,...,0.000184,0.000401,0.00093,0.000894,0.996821,0.996505,1.9,3.1,3.4125,3.063636
7,UCO39187.1,0,M,M,0.0,50,1.0,1,0.04,1.0,...,0.000184,0.000401,0.00093,0.000894,0.996821,0.996505,1.9,3.1,3.4125,3.063636
8,UGO67261.1,0,M,M,0.0,50,1.0,1,0.04,1.0,...,0.000184,0.000401,0.00093,0.000894,0.996821,0.996505,1.9,3.1,3.4125,3.063636
9,UAV83603.1,0,M,M,0.0,50,1.0,1,0.04,1.0,...,0.000184,0.000401,0.00093,0.000894,0.996821,0.996505,1.9,3.1,3.4125,3.063636


In [362]:
#Physiochemical Properties

#create dictionary
chem_prop = {

    #hydrophobic
    "A":"hydrophobic","V":"hydrophobic","L":"hydrophobic",
    "I":"hydrophobic","M": "hydrophobic","F":"hydrophobic",
    "W":"hydrophobic","Y":"hydrophobic",

    #polar
    "S":"polar","T":"polar","N":"polar","Q":"polar",

    #positive:
    "K":"positive","R":"positive","H":"positive",

    #negative:
    "D":"negative","E":"negative",

    #special
    "G":"special","C":"special","P":"special"
    
}

#apply to ref sequence only 
seq1_df["chem_prop"] = seq1_df["wuhan_residue"].map(chem_prop)

#dummy encode each chemical property
chem_prop_df = pd.get_dummies(seq1_df["chem_prop"],dtype = int)

#join it onto dataframe 
seq1_df = seq1_df.join(chem_prop_df)

In [343]:
seq1_df.head(10)

,seq_id,aln_index,sample_residue,wuhan_residue,mutation,valid_n,valid_frac,hotspot,mutation_rate,wuhan_resnum,...,kd_ref,kd_win9,kd_win15,kd_win21,chem_prop,hydrophobic,negative,polar,positive,special
0,UYE19200.1,0,M,M,0.0,50,1.0,1,0.04,1.0,...,1.9,3.1,3.4125,3.063636,hydrophobic,1,0,0,0,0
1,UHX08132.1,0,M,M,0.0,50,1.0,1,0.04,1.0,...,1.9,3.1,3.4125,3.063636,hydrophobic,1,0,0,0,0
2,UHS39707.1,0,M,M,0.0,50,1.0,1,0.04,1.0,...,1.9,3.1,3.4125,3.063636,hydrophobic,1,0,0,0,0
3,XPK52150.1,0,M,M,0.0,50,1.0,1,0.04,1.0,...,1.9,3.1,3.4125,3.063636,hydrophobic,1,0,0,0,0
4,UBO31061.1,0,M,M,0.0,50,1.0,1,0.04,1.0,...,1.9,3.1,3.4125,3.063636,hydrophobic,1,0,0,0,0
5,WPM52290.1,0,-,M,2.0,50,1.0,1,0.04,1.0,...,1.9,3.1,3.4125,3.063636,hydrophobic,1,0,0,0,0
6,UMY93497.1,0,M,M,0.0,50,1.0,1,0.04,1.0,...,1.9,3.1,3.4125,3.063636,hydrophobic,1,0,0,0,0
7,UCO39187.1,0,M,M,0.0,50,1.0,1,0.04,1.0,...,1.9,3.1,3.4125,3.063636,hydrophobic,1,0,0,0,0
8,UGO67261.1,0,M,M,0.0,50,1.0,1,0.04,1.0,...,1.9,3.1,3.4125,3.063636,hydrophobic,1,0,0,0,0
9,UAV83603.1,0,M,M,0.0,50,1.0,1,0.04,1.0,...,1.9,3.1,3.4125,3.063636,hydrophobic,1,0,0,0,0


In [363]:
# Local Net Charge 
charge_df = (
    seq1_df
    .sort_values("wuhan_resnum")
    .drop_duplicates(subset = "wuhan_resnum")
    [["wuhan_resnum", "wuhan_residue"]]
    .copy()
)

charge_map = {"K": 1, "R": 1, "H": 1, "D": -1, "E": -1}

def charge_window(df, windows=[9,15,21]):
    d = df.sort_values("wuhan_resnum").copy()
    aa = d["wuhan_residue"].fillna("").astype(str).str.upper()

    d["charge_ref"] = aa.map(charge_map).fillna(0).astype(float)

    for W in windows:
        d[f"charge_win{W}"] = (
            d["charge_ref"].rolling(window=W, center=True, min_periods=1).mean()
    )
    return d
charge_pos = charge_window(charge_df, windows=[9, 15, 21])

seq1_df = (
    seq1_df
    .drop(columns=["charge_ref","charge_win9","charge_win15","charge_win21",
                   ], errors="ignore")
    .merge(charge_pos[["wuhan_resnum","charge_ref","charge_win9","charge_win15","charge_win21"]],
           on="wuhan_resnum", 
           how="left")
)

In [345]:
print(seq1_df["hotspot"].value_counts()) # 4400/50 = 88 hotspots
seq1_df.head(10)

,seq_id,aln_index,sample_residue,wuhan_residue,mutation,valid_n,valid_frac,hotspot,mutation_rate,wuhan_resnum,...,chem_prop,hydrophobic,negative,polar,positive,special,charge_ref,charge_win9,charge_win15,charge_win21
0,UYE19200.1,0,M,M,0.0,50,1.0,1,0.04,1.0,...,hydrophobic,1,0,0,0,0,0.0,0.0,0.0,0.0
1,UHX08132.1,0,M,M,0.0,50,1.0,1,0.04,1.0,...,hydrophobic,1,0,0,0,0,0.0,0.0,0.0,0.0
2,UHS39707.1,0,M,M,0.0,50,1.0,1,0.04,1.0,...,hydrophobic,1,0,0,0,0,0.0,0.0,0.0,0.0
3,XPK52150.1,0,M,M,0.0,50,1.0,1,0.04,1.0,...,hydrophobic,1,0,0,0,0,0.0,0.0,0.0,0.0
4,UBO31061.1,0,M,M,0.0,50,1.0,1,0.04,1.0,...,hydrophobic,1,0,0,0,0,0.0,0.0,0.0,0.0
5,WPM52290.1,0,-,M,2.0,50,1.0,1,0.04,1.0,...,hydrophobic,1,0,0,0,0,0.0,0.0,0.0,0.0
6,UMY93497.1,0,M,M,0.0,50,1.0,1,0.04,1.0,...,hydrophobic,1,0,0,0,0,0.0,0.0,0.0,0.0
7,UCO39187.1,0,M,M,0.0,50,1.0,1,0.04,1.0,...,hydrophobic,1,0,0,0,0,0.0,0.0,0.0,0.0
8,UGO67261.1,0,M,M,0.0,50,1.0,1,0.04,1.0,...,hydrophobic,1,0,0,0,0,0.0,0.0,0.0,0.0
9,UAV83603.1,0,M,M,0.0,50,1.0,1,0.04,1.0,...,hydrophobic,1,0,0,0,0,0.0,0.0,0.0,0.0


In [364]:
# Glycine and Proline Density 

gp_df = (
    seq1_df
    .sort_values("wuhan_resnum")
    .drop_duplicates(subset = "wuhan_resnum")
    [["wuhan_resnum", "wuhan_residue"]]
    .copy()
)

def gp_windows(df, windows=[9,15,21]):
    d = df.sort_values("wuhan_resnum").copy()
    aa = d["wuhan_residue"].fillna("").astype(str).str.upper()

    # 1 if Gly or Pro else 0
    d["gp_ref"] = aa.isin(["G", "P"]).astype(float)

    for W in windows:
        d[f"gp_win{W}"] = (
            d["gp_ref"]
            .rolling(window=W, center=True, min_periods=1)
            .mean()
        )

    return d
gp_pos = gp_windows(gp_df, windows=[9, 15, 21])

seq1_df = (
    seq1_df
    .merge(gp_pos[["wuhan_resnum","gp_ref","gp_win9","gp_win15","gp_win21"]],
           on="wuhan_resnum", 
           how="left")
)

In [347]:
seq1_df.head(10)

,seq_id,aln_index,sample_residue,wuhan_residue,mutation,valid_n,valid_frac,hotspot,mutation_rate,wuhan_resnum,...,positive,special,charge_ref,charge_win9,charge_win15,charge_win21,gp_ref,gp_win9,gp_win15,gp_win21
0,UYE19200.1,0,M,M,0.0,50,1.0,1,0.04,1.0,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.090909
1,UHX08132.1,0,M,M,0.0,50,1.0,1,0.04,1.0,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.090909
2,UHS39707.1,0,M,M,0.0,50,1.0,1,0.04,1.0,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.090909
3,XPK52150.1,0,M,M,0.0,50,1.0,1,0.04,1.0,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.090909
4,UBO31061.1,0,M,M,0.0,50,1.0,1,0.04,1.0,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.090909
5,WPM52290.1,0,-,M,2.0,50,1.0,1,0.04,1.0,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.090909
6,UMY93497.1,0,M,M,0.0,50,1.0,1,0.04,1.0,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.090909
7,UCO39187.1,0,M,M,0.0,50,1.0,1,0.04,1.0,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.090909
8,UGO67261.1,0,M,M,0.0,50,1.0,1,0.04,1.0,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.090909
9,UAV83603.1,0,M,M,0.0,50,1.0,1,0.04,1.0,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.090909


In [365]:
seq1_df.to_csv("seq1_df.csv",index = False)

## Final Dataset:

In [374]:
#copy to final dataset to use
finalseq_df = seq1_df.copy()
finalseq_df["valid_n"].value_counts()

valid_n
50    28350
48    12650
47     5850
49     5750
46     4750
45     3000
44     2000
43     1150
41      100
42       50
Name: count, dtype: int64

In [375]:
finalseq_df.columns

Index(['seq_id', 'aln_index', 'sample_residue', 'wuhan_residue', 'mutation',
       'valid_n', 'valid_frac', 'hotspot', 'mutation_rate', 'wuhan_resnum',
       'rsa', 'asa', 'q8', 'p[q8_G]', 'p[q8_H]', 'p[q8_I]', 'p[q8_B]',
       'p[q8_E]', 'p[q8_S]', 'p[q8_T]', 'p[q8_C]', 'disorder', 'kd_ref',
       'kd_win9', 'kd_win15', 'kd_win21', 'chem_prop', 'hydrophobic',
       'negative', 'polar', 'positive', 'special', 'charge_ref', 'charge_win9',
       'charge_win15', 'charge_win21', 'gp_ref', 'gp_win9', 'gp_win15',
       'gp_win21'],
      dtype='str')

In [376]:
finalseq_df.groupby("wuhan_resnum")["seq_id"].nunique()

#create position level data frame for normalization not sequence x position since it repeats 50 times
finalseq_df.groupby("wuhan_resnum").first().reset_index()

#remove exploratory columns
finalseq_df = finalseq_df.drop_duplicates(subset = "wuhan_resnum").drop(columns=["seq_id","sample_residue","wuhan_residue","aln_index","mutation","valid_n","valid_frac","mutation_rate","chem_prop"])



print(len(finalseq_df),len(seq1_df)) #1273 63650

1273 63650


In [377]:
finalseq_df

,hotspot,wuhan_resnum,rsa,asa,q8,p[q8_G],p[q8_H],p[q8_I],p[q8_B],p[q8_E],...,positive,special,charge_ref,charge_win9,charge_win15,charge_win21,gp_ref,gp_win9,gp_win15,gp_win21
0,1,1.0,0.660813,148.022039,C,0.000175,0.000576,0.000018,0.000184,0.000401,...,0,0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.090909
50,1,2.0,0.682652,163.836508,C,0.000101,0.000402,0.000006,0.000074,0.000237,...,0,0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.111111,0.083333
100,1,3.0,0.672084,116.942663,C,0.000074,0.000361,0.000003,0.000033,0.000152,...,0,0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.100000,0.076923
150,1,4.0,0.665079,159.619060,C,0.000071,0.000366,0.000002,0.000030,0.000120,...,0,0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.090909,0.071429
200,1,5.0,0.662311,133.124503,C,0.000074,0.000376,0.000003,0.000034,0.000125,...,0,0,0.0,0.000000,0.000000,0.000000,0.0,0.111111,0.083333,0.066667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63400,0,1269.0,0.591457,139.583826,C,0.001593,0.003260,0.000080,0.004701,0.017458,...,1,0,1.0,0.333333,0.166667,0.000000,0.0,0.111111,0.166667,0.133333
63450,0,1270.0,0.422470,84.916435,C,0.002012,0.004981,0.000086,0.003994,0.015040,...,0,0,0.0,0.375000,0.272727,0.071429,0.0,0.125000,0.181818,0.142857
63500,0,1271.0,0.593099,132.854176,C,0.002135,0.005038,0.000123,0.002223,0.008631,...,1,0,1.0,0.285714,0.300000,0.153846,0.0,0.142857,0.100000,0.153846
63550,0,1272.0,0.647031,170.169265,C,0.001559,0.004458,0.000084,0.001535,0.005747,...,0,0,0.0,0.333333,0.333333,0.166667,0.0,0.000000,0.111111,0.166667


In [378]:
finalseq_df.to_csv("finalseq_df.csv",index = False)